# Lesson 0A: Reinforcement Learning Introduction - Theory

## Introduction

Reinforcement Learning (RL) is fundamentally different from supervised and unsupervised learning. Instead of being told the correct answers (supervised) or finding hidden patterns (unsupervised), an RL agent **learns by interacting with an environment**—taking actions, receiving rewards, and discovering the best strategy through trial and error.

Think about learning to play chess:
- **Supervised learning** would require labeled examples: "In this position, the best move is Nf3"
- **Unsupervised learning** would find patterns in past games without any goal
- **Reinforcement learning** learns by playing—winning gives reward (+1), losing gives punishment (-1), and the agent figures out which moves lead to victory

In this lesson, we'll:
1. Understand what RL is and how it differs from other learning paradigms
2. Learn the agent-environment interaction loop
3. Formally define key concepts: states, actions, rewards, policies
4. Explore the exploration vs. exploitation tradeoff
5. Classify RL algorithms: model-based vs. model-free, value-based vs. policy-based
6. See real-world applications (games, robotics, recommendation)
7. Implement and visualize a simple GridWorld environment

Then in Lesson 0B, we'll:
1. Set up Gymnasium (the standard RL environment library)
2. Learn the reset(), step(), and render() APIs
3. Build custom environments
4. Master visualization and debugging techniques

## Table of Contents

1. [Required Libraries](#required-libraries)
2. [Three Paradigms of Machine Learning](#three-paradigms)
3. [The Agent-Environment Interaction Loop](#interaction-loop)
4. [Key Concepts](#key-concepts)
   - [States](#states)
   - [Actions](#actions)
   - [Rewards](#rewards)
   - [Policies](#policies)
   - [Returns and Value Functions](#returns-value)
5. [Exploration vs. Exploitation](#exploration-exploitation)
6. [RL Algorithm Taxonomy](#taxonomy)
   - [Model-based vs. Model-free](#model-based-vs-free)
   - [Value-based vs. Policy-based](#value-vs-policy)
7. [Real-World Applications](#applications)
8. [Simple GridWorld Implementation](#gridworld)
9. [Conclusion and Next Steps](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

| Library | Purpose |
|---------|----------|
| NumPy | Numerical computing and array operations |
| Matplotlib | Visualization of environments and learning curves |
| Seaborn | Statistical visualization |
| Gymnasium | Standard RL environment API (formerly OpenAI Gym) |

In [ ]:
# Install Gymnasium if needed
import subprocess
import sys

try:
    import gymnasium
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium", "-q"])
    print("✅ Gymnasium installed!")

# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
from typing import Dict, List, Tuple, Optional
from matplotlib.patches import Rectangle, FancyArrowPatch
from matplotlib.patches import Circle

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ All libraries loaded!")
print(f"Gymnasium version: {gym.__version__}")

<a name="three-paradigms"></a>
## Three Paradigms of Machine Learning

### Supervised Learning
- **Setup**: We have labeled training data: (x₁, y₁), (x₂, y₂), ..., (xₙ, yₙ)
- **Goal**: Learn a function f(x) that predicts y from x
- **Learning Signal**: The correct label y tells us if we're right or wrong
- **Examples**: Email classification, house price prediction, image recognition

### Unsupervised Learning
- **Setup**: We have unlabeled data: x₁, x₂, ..., xₙ (no y values)
- **Goal**: Find hidden structure or patterns in the data
- **Learning Signal**: No explicit feedback; we define a pattern objective
- **Examples**: Clustering customers, dimensionality reduction, anomaly detection

### Reinforcement Learning
- **Setup**: An agent interacts with an environment by taking actions
- **Goal**: Learn a policy (strategy) that maximizes cumulative reward
- **Learning Signal**: Sparse reward signal; agent must figure out which actions led to reward
- **Examples**: Game playing (chess, Go, Atari), robot control, resource allocation
- **Unique Challenge**: The agent's actions affect what data it sees next (active learning)

In [ ]:
# Visualize the three paradigms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Supervised Learning
ax = axes[0]
ax.text(0.5, 0.9, 'Supervised Learning', ha='center', fontsize=14, fontweight='bold')
ax.text(0.5, 0.75, 'Input: Labeled Data', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightblue'))
ax.text(0.5, 0.6, '↓', ha='center', fontsize=16)
ax.text(0.5, 0.5, 'Train Model', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightgreen'))
ax.text(0.5, 0.35, '↓', ha='center', fontsize=16)
ax.text(0.5, 0.25, 'Predict y from x', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Unsupervised Learning
ax = axes[1]
ax.text(0.5, 0.9, 'Unsupervised Learning', ha='center', fontsize=14, fontweight='bold')
ax.text(0.5, 0.75, 'Input: Unlabeled Data', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightblue'))
ax.text(0.5, 0.6, '↓', ha='center', fontsize=16)
ax.text(0.5, 0.5, 'Find Patterns', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightgreen'))
ax.text(0.5, 0.35, '↓', ha='center', fontsize=16)
ax.text(0.5, 0.25, 'Cluster / Reduce', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Reinforcement Learning
ax = axes[2]
ax.text(0.5, 0.9, 'Reinforcement Learning', ha='center', fontsize=14, fontweight='bold')
ax.text(0.5, 0.75, 'Agent Takes Action', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightblue'))
ax.text(0.5, 0.6, '↓', ha='center', fontsize=16)
ax.text(0.5, 0.5, 'Receives Reward', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightcoral'))
ax.text(0.5, 0.35, '↓', ha='center', fontsize=16)
ax.text(0.5, 0.2, 'Learn Best Policy', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.annotate('', xy=(0.5, 0.15), xytext=(0.5, 0.05),
            arrowprops=dict(arrowstyle='->', lw=2, color='red', linestyle='--'))
ax.text(0.5, 0.02, 'Loop', ha='center', fontsize=10, style='italic', color='red')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

print("Key Insight: RL learns through **interaction**. The agent's actions determine what it sees next.")

<a name="interaction-loop"></a>
## The Agent-Environment Interaction Loop

This is the **heart of reinforcement learning**:

```
┌─────────────────────────────────────────┐
│        AGENT                ENVIRONMENT │
│                                         │
│  1. Observe state: s_t          State   │
│         ↓                                │
│  2. Select action: a_t  ───────────→   │
│                          Step(a_t)      │
│         ↓                        ↓       │
│  3. Receive reward: r_t+1 ←──────      │
│  4. Observe next state: s_t+1 ←──────  │
│         ↓                               │
│  5. Update policy                      │
│         ↓                               │
│    (Loop back to step 1)                │
└─────────────────────────────────────────┘
```

This repeats until the episode ends or the agent's lifetime expires.

<a name="key-concepts"></a>
## Key Concepts

<a name="states"></a>
### 1. States (S)

A **state** is a representation of the current situation the agent observes.

**Examples:**
- Chess: The positions of all pieces on the board
- Robot navigation: Position and velocity of the robot
- Game playing: Current game screen (pixels)
- Resource allocation: Available resources and current demand

**State space**: The set of all possible states S = {s₁, s₂, ..., sₙ}
- **Discrete**: Finite number of states (e.g., chess board has ~10⁴⁷ legal positions)
- **Continuous**: Infinite number of states (e.g., robot position in continuous space)

<a name="actions"></a>
### 2. Actions (A)

An **action** is a choice the agent can make in a given state.

**Examples:**
- Chess: Move a piece (e.g., move knight to e4)
- Robot: Move forward, turn left, turn right
- Game: Up, down, left, right, jump

**Action space**: The set of all possible actions A = {a₁, a₂, ..., aₘ}
- **Discrete**: Finite number of actions (e.g., 4 directions)
- **Continuous**: Infinite number of actions (e.g., wheel torque can be any value in [-1, 1])

<a name="rewards"></a>
### 3. Rewards (R)

A **reward** is a scalar signal that tells the agent how well it's doing.
- **Positive reward** (+): Good action, move toward the goal
- **Negative reward** (-): Bad action, move away from the goal or waste resources
- **Zero reward**: Neutral action

**Examples:**
- Chess: +1 for winning, -1 for losing, 0 for each move
- Robot reaching a goal: +100 for reaching the target, -0.1 per step (penalize time)
- Game: +10 for collecting a coin, -5 for hitting an obstacle

**Critical insight**: Rewards are often **sparse** and **delayed**. The agent must learn which early actions led to late rewards—this is a fundamental challenge in RL.

<a name="policies"></a>
### 4. Policies (π)

A **policy** is the agent's strategy—a mapping from states to actions.

**Deterministic policy:**
- π(s) = a (In state s, always take action a)
- Example: "If you see an opponent's queen undefended, capture it"

**Stochastic policy:**
- π(a|s) = P(a|s) (In state s, take action a with probability p)
- Example: "In state s, take action a with 70% probability, action b with 30% probability"
- Why stochastic? Exploration, randomness in the environment, mixed strategies in games

**Optimal policy π*:**
- The policy that maximizes expected cumulative reward
- Finding π* is the central goal of reinforcement learning

<a name="returns-value"></a>
### 5. Returns and Value Functions

#### Returns (G)

The **return** is the cumulative reward from time t onward:

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + ... = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

Where:
- γ (gamma) is the **discount factor** (0 ≤ γ ≤ 1)
  - γ = 0: Only immediate reward matters (myopic)
  - γ = 1: All future rewards matter equally
  - γ = 0.99: A reward 100 steps in the future is worth 0.99¹⁰⁰ ≈ 37% of its immediate value

**Why discount future rewards?**
1. Mathematical convenience: Makes infinite horizons finite
2. Uncertainty: The farther in the future, the less certain we are about the outcome
3. Time value of money: A reward now is more valuable than a reward later
4. Biological intuition: Living organisms evolved to care more about immediate survival

#### Value Functions

**State value function V(s):**
- The expected return from state s, following policy π
- V(s) = 𝔼[Gₜ | sₜ = s] = Expected cumulative reward starting from state s

**Action value function Q(s,a):**
- The expected return from taking action a in state s, then following policy π
- Q(s,a) = 𝔼[Gₜ | sₜ = s, aₜ = a] = Expected cumulative reward from (s,a) pair

**Key insight:** If we know Q(s,a) for all state-action pairs, we can choose the optimal action:
- π*(s) = argmax_a Q(s,a) (Choose the action with highest value)

In [ ]:
# Visualize return calculation and discount factor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Discount factor visualization
ax = axes[0]
steps = np.arange(0, 101)
gammas = [0.9, 0.95, 0.99, 1.0]

for gamma in gammas:
    discount_curve = gamma ** steps
    ax.plot(steps, discount_curve, linewidth=2.5, label=f'γ = {gamma}')

ax.set_xlabel('Steps into the Future', fontsize=12, fontweight='bold')
ax.set_ylabel('Discount Factor (γ^t)', fontsize=12, fontweight='bold')
ax.set_title('How Discount Factor Affects Future Rewards', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])

# Return calculation example
ax = axes[1]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Title
ax.text(5, 9.5, 'Return Calculation Example', ha='center', fontsize=13, fontweight='bold')
ax.text(5, 8.8, 'Sequence of rewards: r₁=0, r₂=1, r₃=2, r₄=-1, r₅=0, ...', 
        ha='center', fontsize=11, family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# With γ=1
ax.text(1, 7.5, 'With γ = 1.0 (no discounting):', fontsize=11, fontweight='bold')
ax.text(1, 6.8, 'G = 0 + 1 + 2 + (-1) + 0 + ...', fontsize=10, family='monospace')
ax.text(1, 6.2, '= Sum of all future rewards', fontsize=10, style='italic')
ax.text(1, 5.5, 'Problem: Can be infinite!', fontsize=10, color='red', fontweight='bold')

# With γ=0.9
ax.text(1, 4.3, 'With γ = 0.9 (discounted):', fontsize=11, fontweight='bold')
ax.text(1, 3.6, 'G = 0 + 0.9(1) + 0.81(2) + 0.729(-1) + ...', fontsize=10, family='monospace')
ax.text(1, 3.0, '= 0 + 0.9 + 1.62 - 0.729 + ...', fontsize=10, family='monospace')
ax.text(1, 2.3, '≈ 1.79', fontsize=10, fontweight='bold', 
        bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.text(1, 1.6, '✅ Finite! Near rewards weighted more', fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

print("Key Insight: Discount factor γ balances immediate and future rewards.")
print("Higher γ → care about distant future")
print("Lower γ → focus on immediate rewards")

<a name="exploration-exploitation"></a>
## Exploration vs. Exploitation

This is a fundamental tradeoff in RL:

### Exploitation
- Take the action you believe is best based on current knowledge
- Maximize immediate reward
- Risk: Miss better actions you haven't discovered yet
- Example: Always order coffee from your favorite café

### Exploration
- Try new actions to learn if they're better
- Accept short-term lower reward to discover long-term better strategies
- Risk: Waste time on bad actions
- Example: Try a new café to see if it's better

### Strategies

**ε-greedy:**
- With probability ε: Take a random action (explore)
- With probability (1-ε): Take the best known action (exploit)
- Simple and effective; ε typically 0.1 to 0.3, decay over time

**Softmax (Boltzmann):**
- Choose actions with probability proportional to their value
- More nuanced than ε-greedy

**UCB (Upper Confidence Bound):**
- Balance expected reward vs. uncertainty
- Prefer actions we're uncertain about

**Thompson Sampling:**
- Maintain a posterior distribution over action values
- Sample from posterior and act greedily
- Theoretically optimal but computationally expensive

In [ ]:
# Simulate exploration vs. exploitation in a simple multi-armed bandit problem
def simulate_bandit(true_means, strategy='greedy', epsilon=0.1, n_trials=1000):
    """
    Simulate a multi-armed bandit problem.
    
    Args:
        true_means: List of true reward means for each arm
        strategy: 'greedy', 'epsilon_greedy', or 'optimal'
        epsilon: Exploration rate for epsilon-greedy
        n_trials: Number of trials
    """
    n_arms = len(true_means)
    action_counts = np.zeros(n_arms)
    q_estimates = np.zeros(n_arms)  # Estimated Q values
    rewards = []
    actions_taken = []
    
    for trial in range(n_trials):
        # Choose action
        if strategy == 'greedy':
            action = np.argmax(q_estimates)
        elif strategy == 'epsilon_greedy':
            if np.random.random() < epsilon:
                action = np.random.randint(n_arms)  # Explore
            else:
                action = np.argmax(q_estimates)  # Exploit
        elif strategy == 'optimal':
            action = np.argmax(true_means)  # Know true means (cheating!)
        
        # Get reward
        reward = np.random.normal(true_means[action], 1.0)
        
        # Update estimates
        action_counts[action] += 1
        q_estimates[action] += (reward - q_estimates[action]) / action_counts[action]
        
        # Record
        rewards.append(reward)
        actions_taken.append(action)
    
    return rewards, actions_taken, q_estimates, action_counts

# Run simulations
true_means = [1.0, 2.0, 3.5]  # True reward means; arm 2 is best

greedy_rewards, _, _, greedy_counts = simulate_bandit(true_means, 'greedy', n_trials=1000)
epsilon_rewards, epsilon_actions, _, epsilon_counts = simulate_bandit(true_means, 'epsilon_greedy', epsilon=0.1, n_trials=1000)
optimal_rewards, _, _, optimal_counts = simulate_bandit(true_means, 'optimal', n_trials=1000)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cumulative reward
ax = axes[0, 0]
ax.plot(np.cumsum(greedy_rewards), label='Pure Greedy', linewidth=2, alpha=0.7)
ax.plot(np.cumsum(epsilon_rewards), label='ε-Greedy (ε=0.1)', linewidth=2, alpha=0.7)
ax.plot(np.cumsum(optimal_rewards), label='Optimal (cheating)', linewidth=2, alpha=0.7, linestyle='--')
ax.set_xlabel('Trial', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative Reward', fontsize=11, fontweight='bold')
ax.set_title('Cumulative Reward Over Time', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Action counts (exploration pattern)
ax = axes[0, 1]
x = np.arange(len(true_means))
width = 0.35
ax.bar(x - width/2, greedy_counts, width, label='Pure Greedy', alpha=0.8)
ax.bar(x + width/2, epsilon_counts, width, label='ε-Greedy', alpha=0.8)
ax.set_xlabel('Arm', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Times Chosen', fontsize=11, fontweight='bold')
ax.set_title('How Many Times Each Arm Was Chosen', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Arm {i}\n(μ={m})' for i, m in enumerate(true_means)])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Moving average reward
ax = axes[1, 0]
window = 50
greedy_ma = np.convolve(greedy_rewards, np.ones(window)/window, mode='valid')
epsilon_ma = np.convolve(epsilon_rewards, np.ones(window)/window, mode='valid')
optimal_ma = np.convolve(optimal_rewards, np.ones(window)/window, mode='valid')

ax.plot(greedy_ma, label='Pure Greedy', linewidth=2, alpha=0.7)
ax.plot(epsilon_ma, label='ε-Greedy (ε=0.1)', linewidth=2, alpha=0.7)
ax.plot(optimal_ma, label='Optimal', linewidth=2, alpha=0.7, linestyle='--')
ax.set_xlabel('Trial', fontsize=11, fontweight='bold')
ax.set_ylabel(f'Average Reward (window={window})', fontsize=11, fontweight='bold')
ax.set_title('Average Reward: Who Learns Best?', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Analysis text
ax = axes[1, 1]
ax.axis('off')
ax.text(0.5, 0.95, 'Key Findings', ha='center', fontsize=12, fontweight='bold',
        transform=ax.transAxes)

findings = f"""Pure Greedy:
• Gets stuck on first arm found
• Achieves ~{greedy_counts[2]:.0f} pulls of best arm
• Final reward ≈ {greedy_rewards[-1]:.2f}
• Gets unlucky early → converges slow

ε-Greedy (ε=0.1):
• Explores all arms
• Achieves ~{epsilon_counts[2]:.0f} pulls of best arm
• Final reward ≈ {epsilon_rewards[-1]:.2f}
• Finds best arm reliably!

Optimal (Cheating):
• Always pulls best arm
• Achieves {optimal_counts[2]:.0f} pulls of best arm
• Final reward ≈ {optimal_rewards[-1]:.2f}
• Benchmark

Conclusion:
Exploration enables learning!
"""

ax.text(0.05, 0.85, findings, ha='left', va='top', fontsize=10, family='monospace',
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

<a name="taxonomy"></a>
## RL Algorithm Taxonomy

RL algorithms are classified along two dimensions:

<a name="model-based-vs-free"></a>
### 1. Model-Based vs. Model-Free

**Model-Based RL:**
- Learn/use a **model** of the environment: P(s'|s,a) = probability of next state, R(s,a) = expected reward
- Plan: Use the model to simulate future trajectories
- Examples: Dynamic Programming (policy iteration, value iteration), Monte Carlo Tree Search (AlphaGo)
- **Pros**: Efficient learning, can evaluate actions before trying them
- **Cons**: Model may be wrong; environment may be complex to model

**Model-Free RL:**
- Learn directly from experience without building a model
- No planning; just learn value function or policy
- Examples: Q-Learning, Policy Gradients, Actor-Critic
- **Pros**: Simpler, works even if environment is unpredictable
- **Cons**: Sample inefficient; needs lots of interaction

<a name="value-vs-policy"></a>
### 2. Value-Based vs. Policy-Based

**Value-Based Methods:**
- Learn the value function V(s) or Q(s,a)
- Derive policy greedily: π(s) = argmax_a Q(s,a)
- Examples: Q-Learning, Temporal Difference Learning
- **Best for**: Discrete action spaces, simpler problems

**Policy-Based Methods:**
- Directly learn the policy π(a|s)
- Optimize policy using gradient methods
- Examples: Policy Gradients (REINFORCE), Trust Region Methods (TRPO, PPO)
- **Best for**: Continuous action spaces, complex policies, stochastic environments

**Actor-Critic (Hybrid):**
- Learn both value function (critic) and policy (actor)
- Actor (policy) decides what to do
- Critic (value) provides feedback on quality
- Examples: A3C, SAC, TD3
- **Best for**: Large state/action spaces, continuous control

In [ ]:
# Visualize the RL algorithm taxonomy
fig, ax = plt.subplots(figsize=(14, 8))

# Draw the matrix
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)
ax.set_aspect('equal')

# Draw grid
for i in range(3):
    ax.axhline(i, color='black', linewidth=2)
    ax.axvline(i, color='black', linewidth=2)

# Labels
ax.text(1.5, 2.3, 'Model-Based vs. Model-Free & Value-Based vs. Policy-Based', 
        ha='center', fontsize=14, fontweight='bold')

# Row labels (Value vs Policy)
ax.text(-0.35, 1.5, 'Value-Based', ha='right', va='center', fontsize=12, fontweight='bold', rotation=90)
ax.text(-0.35, 0.5, 'Policy-Based', ha='right', va='center', fontsize=12, fontweight='bold', rotation=90)

# Column labels (Model-Based vs Free)
ax.text(1.5, 2.35, 'Model-Based', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.text(1.5, 2.35, 'Model-Free', ha='center', va='bottom', fontsize=12, fontweight='bold', x=0.5)

# Cell 1: Model-Based + Value-Based
ax.text(1.5, 1.85, 'Dynamic Programming', ha='center', fontsize=11, fontweight='bold')
ax.text(1.5, 1.65, 'Policy Iteration', ha='center', fontsize=10)
ax.text(1.5, 1.5, 'Value Iteration', ha='center', fontsize=10)
ax.text(1.5, 1.35, '• Requires model P,R', ha='center', fontsize=9, style='italic')
ax.text(1.5, 1.2, '• Guaranteed optimal', ha='center', fontsize=9, style='italic')
ax.text(1.5, 1.05, '• Slow for large state spaces', ha='center', fontsize=9, style='italic')

# Cell 2: Model-Free + Value-Based
ax.text(0.5, 1.85, 'Temporal Difference', ha='center', fontsize=11, fontweight='bold')
ax.text(0.5, 1.65, 'Q-Learning', ha='center', fontsize=10)
ax.text(0.5, 1.5, 'Sarsa', ha='center', fontsize=10)
ax.text(0.5, 1.35, '• Learn from experience', ha='center', fontsize=9, style='italic')
ax.text(0.5, 1.2, '• No model required', ha='center', fontsize=9, style='italic')
ax.text(0.5, 1.05, '• Work well for discrete spaces', ha='center', fontsize=9, style='italic')

# Cell 3: Model-Based + Policy-Based
ax.text(1.5, 0.85, 'Planning + Search', ha='center', fontsize=11, fontweight='bold')
ax.text(1.5, 0.65, 'Monte Carlo Tree Search', ha='center', fontsize=10)
ax.text(1.5, 0.5, 'AlphaGo (with neural network)', ha='center', fontsize=10)
ax.text(1.5, 0.35, '• Uses learned model for planning', ha='center', fontsize=9, style='italic')
ax.text(1.5, 0.2, '• Sample-efficient', ha='center', fontsize=9, style='italic')

# Cell 4: Model-Free + Policy-Based
ax.text(0.5, 0.85, 'Policy Gradient Methods', ha='center', fontsize=11, fontweight='bold')
ax.text(0.5, 0.65, 'REINFORCE, PPO, A3C, SAC', ha='center', fontsize=10)
ax.text(0.5, 0.5, 'Actor-Critic', ha='center', fontsize=10)
ax.text(0.5, 0.35, '• Direct policy optimization', ha='center', fontsize=9, style='italic')
ax.text(0.5, 0.2, '• Handle continuous actions', ha='center', fontsize=9, style='italic')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

<a name="applications"></a>
## Real-World Applications

### Game Playing
- **AlphaGo (2016)**: Defeated world champion Lee Sedol at Go using deep RL + Monte Carlo Tree Search
- **Atari games**: DQN learned to play Pong, Breakout, Space Invaders from raw pixels
- **Dota 2**: OpenAI Five defeated professional esports team

### Robotics
- **Robotic manipulation**: Learning to grasp and manipulate objects
- **Autonomous navigation**: Drones learning to fly autonomously
- **Robot walking**: Legged robots learning locomotion from scratch

### Resource Allocation
- **Power grid management**: Optimizing electricity distribution
- **Data center cooling**: Learning optimal fan speeds to save energy
- **Traffic signal control**: Optimizing traffic flow in cities

### Finance & Recommendation
- **Portfolio optimization**: Learning which stocks to hold
- **Recommendation systems**: Netflix, YouTube learning what to recommend
- **Bidding strategies**: Ad networks learning optimal bid amounts

### Natural Language Processing
- **RLHF (Reinforcement Learning from Human Feedback)**: Fine-tuning LLMs like ChatGPT
- **Dialog systems**: Learning conversational strategies
- **Machine translation**: Learning translation policies

<a name="gridworld"></a>
## Simple GridWorld Implementation

Let's implement a simple 4×4 GridWorld environment to see RL concepts in action.

**Environment:**
- 4×4 grid (16 possible states)
- Agent starts at top-left
- Goal: Reach bottom-right
- Actions: Up, Down, Left, Right
- Rewards: +1 for reaching goal, -1 for each step (penalize time)

In [ ]:
class GridWorld:
    """Simple 4x4 GridWorld environment."""
    
    def __init__(self, grid_size=4):
        self.grid_size = grid_size
        self.start_pos = (0, 0)
        self.goal_pos = (grid_size - 1, grid_size - 1)
        self.agent_pos = self.start_pos
        self.episode_step = 0
        self.max_steps = 20
        
        # Action space: 0=up, 1=down, 2=left, 3=right
        self.actions = ['up', 'down', 'left', 'right']
        self.action_deltas = {
            'up': (-1, 0),
            'down': (1, 0),
            'left': (0, -1),
            'right': (0, 1)
        }
    
    def reset(self):
        """Reset environment to initial state."""
        self.agent_pos = self.start_pos
        self.episode_step = 0
        return self._get_state()
    
    def step(self, action):
        """Execute one step in the environment.
        
        Args:
            action: 0=up, 1=down, 2=left, 3=right
        
        Returns:
            state: New state
            reward: Reward for this step
            done: Whether episode is finished
            info: Additional info
        """
        # Move agent
        action_name = self.actions[action]
        dr, dc = self.action_deltas[action_name]
        new_r, new_c = self.agent_pos[0] + dr, self.agent_pos[1] + dc
        
        # Check boundaries
        if 0 <= new_r < self.grid_size and 0 <= new_c < self.grid_size:
            self.agent_pos = (new_r, new_c)
        # else: hit boundary, stay in place
        
        self.episode_step += 1
        
        # Calculate reward
        if self.agent_pos == self.goal_pos:
            reward = 1.0  # Reached goal!
            done = True
        elif self.episode_step >= self.max_steps:
            reward = -0.1  # Ran out of time
            done = True
        else:
            reward = -0.01  # Small cost for each step
            done = False
        
        return self._get_state(), reward, done, {}
    
    def _get_state(self):
        """Convert position to state index (0-15)."""
        return self.agent_pos[0] * self.grid_size + self.agent_pos[1]
    
    def render(self, show_grid=True):
        """Visualize the current state."""
        if show_grid:
            for r in range(self.grid_size):
                for c in range(self.grid_size):
                    if (r, c) == self.agent_pos:
                        print("A", end=" ")  # Agent
                    elif (r, c) == self.goal_pos:
                        print("G", end=" ")  # Goal
                    elif (r, c) == self.start_pos:
                        print("S", end=" ")  # Start
                    else:
                        print(".", end=" ")
                print()
            print(f"Step: {self.episode_step}/{self.max_steps}")

# Create environment
env = GridWorld(grid_size=4)
print("GridWorld Environment Created!")
print(f"Grid size: {env.grid_size}×{env.grid_size}")
print(f"Start: {env.start_pos}")
print(f"Goal: {env.goal_pos}")
print(f"Actions: {env.actions}")
print(f"\nInitial state:")
env.render()

In [ ]:
# Let's run a few random trajectories
np.random.seed(42)

print("=" * 50)
print("Random Policy: Taking random actions")
print("=" * 50)

for episode in range(2):
    state = env.reset()
    total_reward = 0
    print(f"\n--- Episode {episode + 1} ---")
    env.render()
    
    for step in range(env.max_steps):
        # Random action
        action = np.random.randint(4)
        state, reward, done, _ = env.step(action)
        total_reward += reward
        
        print(f"\nAction: {env.actions[action]} → Reward: {reward:.3f}")
        env.render()
        
        if done:
            print(f"\n✅ Episode finished in {step + 1} steps with total reward: {total_reward:.3f}")
            if env.agent_pos == env.goal_pos:
                print("🎉 Goal reached!")
            else:
                print("❌ Time limit exceeded")
            break

In [ ]:
# Implement Q-Learning to learn the optimal policy
class QLearningAgent:
    """Q-Learning agent for GridWorld."""
    
    def __init__(self, n_states, n_actions, learning_rate=0.1, discount=0.99, epsilon=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = discount
        self.epsilon = epsilon
        
        # Initialize Q-table
        self.Q = np.zeros((n_states, n_actions))
    
    def get_action(self, state, training=True):
        """ε-greedy action selection."""
        if training and np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)  # Explore
        else:
            return np.argmax(self.Q[state])  # Exploit
    
    def update(self, state, action, reward, next_state, done):
        """Q-Learning update: Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') - Q(s,a)]"""
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.Q[next_state])
        
        td_error = target - self.Q[state, action]
        self.Q[state, action] += self.lr * td_error
    
    def get_policy(self):
        """Extract greedy policy from Q-table."""
        return np.argmax(self.Q, axis=1)

# Train the agent
print("\n" + "=" * 50)
print("Training Q-Learning Agent")
print("=" * 50)

n_states = 16
n_actions = 4

agent = QLearningAgent(n_states, n_actions, learning_rate=0.1, epsilon=0.1)

n_episodes = 200
episode_rewards = []
episode_lengths = []

for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    steps = 0
    
    for step in range(env.max_steps):
        action = agent.get_action(state, training=True)
        next_state, reward, done, _ = env.step(action)
        
        agent.update(state, action, reward, next_state, done)
        
        total_reward += reward
        steps += 1
        state = next_state
        
        if done:
            break
    
    episode_rewards.append(total_reward)
    episode_lengths.append(steps)
    
    if (episode + 1) % 50 == 0:
        avg_reward = np.mean(episode_rewards[-50:])
        avg_length = np.mean(episode_lengths[-50:])
        print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.3f}, Avg Length = {avg_length:.1f}")

print("\n✅ Training complete!")

In [ ]:
# Visualize learning progress
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curve
ax = axes[0]
ax.plot(episode_rewards, alpha=0.6, linewidth=1, label='Episode Reward')
window = 20
ma_reward = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, n_episodes), ma_reward, linewidth=2.5, color='red', label=f'{window}-episode Moving Avg')
ax.set_xlabel('Episode', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Reward', fontsize=12, fontweight='bold')
ax.set_title('Learning Curve: Total Reward per Episode', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Episode length
ax = axes[1]
ax.plot(episode_lengths, alpha=0.6, linewidth=1, label='Episode Length')
ma_length = np.convolve(episode_lengths, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, n_episodes), ma_length, linewidth=2.5, color='red', label=f'{window}-episode Moving Avg')
ax.set_xlabel('Episode', fontsize=12, fontweight='bold')
ax.set_ylabel('Steps to Goal', fontsize=12, fontweight='bold')
ax.set_title('Learning Efficiency: Steps to Reach Goal', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(6, color='green', linestyle='--', linewidth=2, label='Optimal Path (6 steps)', alpha=0.7)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nLearning Summary:")
print(f"Initial avg reward (first 20): {np.mean(episode_rewards[:20]):.3f}")
print(f"Final avg reward (last 20): {np.mean(episode_rewards[-20:]):.3f}")
print(f"Initial avg steps (first 20): {np.mean(episode_lengths[:20]):.1f}")
print(f"Final avg steps (last 20): {np.mean(episode_lengths[-20:]):.1f}")
print(f"Optimal path length: 6 steps")

In [ ]:
# Test the learned policy
print("\n" + "=" * 50)
print("Testing Learned Policy (No Exploration)")
print("=" * 50)

for episode in range(2):
    state = env.reset()
    total_reward = 0
    print(f"\n--- Episode {episode + 1} ---")
    env.render()
    
    for step in range(env.max_steps):
        # Use learned policy (greedy, no exploration)
        action = agent.get_action(state, training=False)
        state, reward, done, _ = env.step(action)
        total_reward += reward
        
        print(f"\nAction: {env.actions[action]} → Reward: {reward:.3f}")
        env.render()
        
        if done:
            print(f"\n✅ Episode finished in {step + 1} steps with total reward: {total_reward:.3f}")
            if env.agent_pos == env.goal_pos:
                print("🎉 Goal reached!")
            break

In [ ]:
# Visualize learned Q-table
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Q-values heatmap
ax = axes[0]
max_q_values = np.max(agent.Q, axis=1).reshape(4, 4)
im = ax.imshow(max_q_values, cmap='RdYlGn', origin='upper')
ax.set_title('Max Q-Value at Each State', fontsize=13, fontweight='bold')
ax.set_xlabel('Column', fontsize=11)
ax.set_ylabel('Row', fontsize=11)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{max_q_values[i, j]:.2f}', ha='center', va='center',
                color='white' if max_q_values[i, j] > max_q_values.max() / 2 else 'black',
                fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, label='Max Q-Value')

# Policy visualization
ax = axes[1]
policy = agent.get_policy()
action_symbols = ['↑', '↓', '←', '→']

for state in range(16):
    r, c = state // 4, state % 4
    action = policy[state]
    
    if (r, c) == env.goal_pos:
        ax.text(c, r, '★', ha='center', va='center', fontsize=20, color='gold')
    else:
        ax.text(c, r, action_symbols[action], ha='center', va='center', fontsize=16, fontweight='bold')

ax.set_xlim(-0.5, 3.5)
ax.set_ylim(3.5, -0.5)
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_title('Learned Policy (Arrows = Actions)', fontsize=13, fontweight='bold')
ax.set_xlabel('Column', fontsize=11)
ax.set_ylabel('Row', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nPolicy Interpretation:")
print("Each cell shows the optimal action to take from that state.")
print("★ = Goal position")
print("The arrows point toward the goal! This is the learned optimal policy.")

<a name="conclusion"></a>
## Conclusion

### What We Learned

1. **RL differs from supervised/unsupervised learning** by learning through interaction with an environment
2. **The agent-environment loop** is the core: observe state → select action → receive reward → update
3. **Key concepts**:
   - States (S): Current situation
   - Actions (A): Choices available
   - Rewards (R): Feedback signal
   - Policies (π): Strategy for choosing actions
   - Value functions: Expected cumulative rewards
4. **Exploration vs. Exploitation**: Balance trying new things vs. using what works
5. **RL taxonomy**:
   - Model-based vs. model-free
   - Value-based vs. policy-based
6. **GridWorld example** showed Q-Learning can discover optimal policies

### Key Insights

- RL is powerful because it learns from interaction, not from labeled data
- Discount factor γ balances immediate and future rewards
- Exploration is essential for learning; pure exploitation gets stuck
- Value functions guide policy learning
- Same fundamental concepts apply to games, robotics, recommendation systems

### Next: Lesson 0B

In the practical notebook, we'll:
1. Set up Gymnasium, the standard RL environment library
2. Learn the environment API: `reset()`, `step()`, `render()`
3. Build custom environments
4. Master visualization and debugging

The foundation is set. You now understand what RL is and why it matters. Let's build! 🚀